# 🏦 Simulación de cajas – Banco de Colombia
**Objetivo:** decidir cuántas cajas deben ser exclusivas para *retiros* y cuántas para *pagos/consignaciones*, y si hace falta una caja nueva.

**Supuestos del modelo**
- 3 cajas, cada una es una cola **M/M/1** independiente (atención inmediata, desplazamiento despreciable).
- El banco opera **8 horas = 480 min**. Se simula 1 día y se repite en **10 réplicas**.
- Cada cliente que llega es de un tipo (Tabla 2: 70 % retiros, 30 % pagos, y dentro de cada acción su probabilidad por tipo de usuario). El tiempo entre llegadas y el tiempo de servicio son exponenciales con las medias de la Tabla 1 para ese tipo.
- Cada cliente elige al azar una de las cajas habilitadas para su servicio (cajas independientes).

In [ ]:
import numpy as np
import pandas as pd

# ---------- Tablas del enunciado ----------
# (acción, tipo): (media servicio, media llegada, probabilidad dentro de la acción)
tabla = {
    ('Retiro','Rápido'):    (1, 1, 0.23),
    ('Retiro','Normal'):    (2, 2, 0.40),
    ('Retiro','Lento'):     (3, 3, 0.17),
    ('Retiro','Muy lento'): (4, 3, 0.20),
    ('Pago','Rápido'):      (3, 1, 0.10),
    ('Pago','Normal'):      (3, 2, 0.20),
    ('Pago','Lento'):       (5, 3, 0.30),
    ('Pago','Muy lento'):   (7, 4, 0.40),
}
P_ACCION = {'Retiro':0.70, 'Pago':0.30}

tipos  = list(tabla.keys())
probs  = np.array([P_ACCION[a]*tabla[(a,t)][2] for a,t in tipos])   # prob. global de cada tipo
mu_srv = np.array([tabla[k][0] for k in tipos], dtype=float)
mu_arr = np.array([tabla[k][1] for k in tipos], dtype=float)
print("Suma de probabilidades:", probs.sum())

T_DIA = 480      # minutos (8 horas)
N_REP = 10       # réplicas

## 1. Funciones de simulación

In [ ]:
def generar_clientes(rng):
    """Genera los clientes de un día: llegada, tipo y tiempo de servicio."""
    t, filas = 0.0, []
    while True:
        k = rng.choice(len(tipos), p=probs)            # tipo del cliente
        t += rng.exponential(mu_arr[k])                # tiempo entre llegadas
        if t > T_DIA: break
        filas.append((t, tipos[k][0], tipos[k][1], rng.exponential(mu_srv[k])))
    return pd.DataFrame(filas, columns=['llegada','accion','tipo','servicio'])

def simular(clientes, config, rng):
    """config = lista con el servicio de cada caja: 'Retiro', 'Pago' o 'Mixta'.
       Cada caja es una cola FIFO M/M/1."""
    df = clientes.copy()
    df['caja'] = 0
    for i, c in df.iterrows():                        # asignación aleatoria a una caja habilitada
        opciones = [j for j, s in enumerate(config) if s in ('Mixta', c.accion)]
        df.at[i, 'caja'] = rng.choice(opciones)
    inicio, fin = [], []
    libre = [0.0]*len(config)                         # momento en que se libera cada caja
    for _, c in df.iterrows():
        j = int(c.caja)
        ini = max(c.llegada, libre[j])
        libre[j] = ini + c.servicio
        inicio.append(ini); fin.append(libre[j])
    df['espera']  = np.array(inicio) - df.llegada
    df['sistema'] = np.array(fin) - df.llegada        # espera + servicio
    return df

# Escenarios a comparar
escenarios = {
    'A: 3 cajas mixtas'         : ['Mixta','Mixta','Mixta'],
    'B: 1 retiro + 2 pagos'     : ['Retiro','Pago','Pago'],
    'C: 2 retiros + 1 pago'     : ['Retiro','Retiro','Pago'],
    'D: 4 cajas mixtas (nueva)' : ['Mixta']*4,
}

## 2. Ejecutar las 10 réplicas
Se usan los mismos clientes en todos los escenarios de cada réplica para comparar de forma justa.

In [ ]:
resultados = {e: [] for e in escenarios}
for r in range(N_REP):
    rng = np.random.default_rng(100 + r)
    clientes = generar_clientes(rng)
    for nombre, cfg in escenarios.items():
        sim = simular(clientes, cfg, np.random.default_rng(r))
        sim['replica'] = r + 1
        resultados[nombre].append(sim)
resultados = {e: pd.concat(v) for e, v in resultados.items()}
print("Listo:", N_REP, "réplicas x", len(escenarios), "escenarios")

## Punto 1 – Cajero con menor y mayor tiempo promedio de atención
Modelo actual: 3 cajas mixtas (escenario A). Tiempo de atención = tiempo de servicio; también se muestra el tiempo total en el sistema.

In [ ]:
A = resultados['A: 3 cajas mixtas']
p1 = A.groupby('caja').agg(clientes=('servicio','size'),
                           atencion_prom=('servicio','mean'),
                           espera_prom=('espera','mean'),
                           sistema_prom=('sistema','mean')).round(2)
p1.index = [f'Caja {i+1}' for i in p1.index]
display(p1)
print("Menor tiempo promedio de atención:", p1.atencion_prom.idxmin(), "->", p1.atencion_prom.min(), "min")
print("Mayor tiempo promedio de atención:", p1.atencion_prom.idxmax(), "->", p1.atencion_prom.max(), "min")

## Punto 2 – Promedio de usuarios de cada tipo (todas las cajas)

In [ ]:
cont = A.groupby(['replica','accion','tipo']).size().unstack(['accion','tipo']).fillna(0)
p2 = cont.mean().round(1).rename('promedio de usuarios por día')
display(p2.to_frame())

## Punto 3 – Total de usuarios por tipo en cada réplica y réplica con menos usuarios

In [ ]:
display(cont.astype(int))
totales = cont.sum(axis=1)
print("Réplica con MENOR cantidad total de usuarios:", int(totales.idxmin()), "->", int(totales.min()), "usuarios")
print("\nRéplica con menor cantidad por cada tipo:")
display(cont.idxmin().rename('réplica').to_frame())

## Punto 4 – ¿Se necesita una cuarta caja?
Se comparan tiempos de espera, tiempo en el sistema y la espera máxima entre 3 y 4 cajas.

In [ ]:
def resumen(df):
    return pd.Series({
        'espera prom (min)'     : df.espera.mean(),
        'tiempo en sistema (min)': df.sistema.mean(),
        'espera máx (min)'      : df.espera.max(),
        '% que espera >5 min'   : (df.espera > 5).mean()*100,
    })
tabla_esc = pd.DataFrame({e: resumen(df) for e, df in resultados.items()}).T.round(2)
display(tabla_esc)

# Tiempo de espera por tipo de acción en cada escenario
por_accion = pd.DataFrame({e: df.groupby('accion').espera.mean() for e, df in resultados.items()}).T.round(2)
print("Espera promedio (min) por tipo de acción:")
display(por_accion)

## Punto 5 – ¿Cuántas cajas exclusivas para pagos y cuántas para retiros?

In [ ]:
cand = tabla_esc.loc[['A: 3 cajas mixtas','B: 1 retiro + 2 pagos','C: 2 retiros + 1 pago']]
excl = cand.loc[['B: 1 retiro + 2 pagos','C: 2 retiros + 1 pago']]
mejor = excl['espera prom (min)'].idxmin()
print("Mejor configuración EXCLUSIVA (menor espera promedio):", mejor)
display(cand)

# Gráfica simple para la exposición
import matplotlib.pyplot as plt
ax = cand['espera prom (min)'].plot(kind='bar', color=['#999','#4C78A8','#F58518'], rot=15, figsize=(7,4))
ax.set_ylabel('Espera promedio (min)'); ax.set_title('Espera promedio por configuración')
plt.tight_layout(); plt.show()

## ✅ Conclusiones (con la semilla del cuaderno; si cambias la semilla los números varían un poco)
1. **Atención por cajero:** la Caja 3 tiene el menor tiempo promedio de atención (~3,03 min) y la Caja 1 el mayor (~3,33 min). La diferencia es pequeña porque los clientes se reparten al azar.
2. **Usuarios promedio por día:** los retiros dominan (~134 por día, sobre todo tipo *Normal*, ~54) frente a ~57 pagos (sobre todo *Muy lento*, ~22).
3. **Réplica con menos usuarios:** la réplica 9 (162 usuarios en total); ver la tabla del punto 3 para el detalle por tipo.
4. **Nueva caja:** con 3 cajas la espera promedio es ~3,0 min y ~19 % de los clientes espera más de 5 min. Con 4 cajas baja a ~1,9 min y ~12 %. Por tanto, **sí conviene crear una cuarta caja** si el banco quiere reducir la espera de forma notable.
5. **Distribución de cajas:** entre las opciones exclusivas, **2 cajas para retiros y 1 para pagos** (espera ~3,4 min) es mejor que 1 retiro y 2 pagos (~4,7 min), porque los retiros son el 70 % de la demanda. Aun así, la espera de pagos sube (~7,9 min) en esa opción, por lo que las 3 cajas mixtas (~3,0 min) siguen siendo la más equilibrada.